<a href="https://colab.research.google.com/github/hanshunyi0-cyber/mlp-s3-case-study/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — European Rain Forecast: a Minimal Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-agent.ipynb)

The smallest agent worth submitting to challenge 177:
for each city and calendar month, the 100 quantiles of hourly rain
over 2020–2026, used as 100 members.

You will build the table, write `agent.py` and submit it. The next
notebook scores agents locally, the way the challenge does.

**Deliverable:** your agent on the leaderboard
of challenge 177.

---

## 1. The contract

Every 6 hours the challenge calls your agent **once**, with the whole
panel: `Agent().predict(request) -> dict`.

| request key | shape | what it is |
|---|---|---|
| `issue_time` | str | ISO UTC hour the forecast is made from |
| `timestamps` | 48 str | the input hours, ascending, ending at `issue_time` |
| `observed` | 48 bool | `False`: the feed dropped that hour, the previous one was carried forward |
| `cities` | 45 dict | `name`, `country`, `lat`, `lon`, in a fixed order |
| `feature_names` | 8 str | `temperature`, `rain`, `wind_speed`, `wind_direction`, `humidity`, `clouds`, `visibility`, `snow` |
| `history` | (45, 48, 8) | cities × hours × features |
| `horizons` | [6, 48] | lead times, hours after `issue_time` |
| `max_members` | 100 | the largest M allowed |

The response is `{"rain": (45, 2, M)}` as nested lists: for each city
and horizon, M plausible amounts of rain in mm at `issue_time + h`.
1 ≤ M ≤ 100, the same M everywhere, every value finite; values below
0 are clipped to 0. Samples or quantiles, both are members. M = 1 is
a deterministic forecast.

Each forecast is scored **once**, 48 hours later, when both hours have
been observed: the CRPS of your members against the rain that fell
(the analysis notebook, section 9), averaged over the 45 cities at
each horizon, and the run scores ½(CRPS₆ + CRPS₄₈) — in mm, lower is
better.

The call has 60 s, 3 CPUs and 3 GiB. Files you upload sit next to
`agent.py`.

---

## 2. Setup and download

The notebook needs one secret, `MLARENA_API_KEY`: ML-Arena, Profile →
API Keys (it starts with `mlk_user_`). Never paste its value into a
cell: a notebook is shared with its code and its outputs.

In Colab, add it to the *Secrets* panel (the key icon on the left) and
allow this notebook to access it. Outside Colab, set it in your
environment before starting Jupyter; the cell raises if it is missing.
Colab already has pandas, numpy and matplotlib; the only install is
the ML-Arena client.

In [1]:
!pip install -q mlarena-sdk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import os
import subprocess
import sys
from google.colab import userdata
MLARENA_API_KEY = userdata.get("MLARENA_API_KEY")

import matplotlib.pyplot as plt
import mlarena
import numpy as np
import pandas as pd
import requests

pd.set_option("display.width", 120)
client = mlarena.connect(api_key=MLARENA_API_KEY)
CHALLENGE_ID = 177

Challenge 177 carries one dataset: the 28 MB European panel
`weather_europe_2020_2026.csv.gz`. The cell below asks for the list,
picks that file by its label and fetches its signed `download_url`
(valid one hour, no key needed); `client.download_dataset(177)` does
the same in one call.

In [3]:
FILE = "weather_europe_2020_2026.csv.gz"

if not os.path.exists(FILE):
    listing = client.datasets(CHALLENGE_ID)
    [meta] = [f for ds in listing["datasets"] for f in ds["files"]
              if f["label"] == FILE]
    resp = requests.get(meta["download_url"], timeout=300)
    resp.raise_for_status()
    with open(FILE, "wb") as fh:
        fh.write(resp.content)
print(FILE, f"{os.path.getsize(FILE) / 1e6:.1f} MB")

weather_europe_2020_2026.csv.gz 28.2 MB


---

## 3. The data, as the challenge sees it

The challenge sends the cities in its own fixed order, not the
file's alphabetical one. `X` is the whole file as an array (hours,
cities, features) in that order; the model reads its rain.

In [4]:
PANEL = [
    ("Amsterdam", "NL"), ("Athens", "GR"), ("Belgrade", "RS"),
    ("Berlin", "DE"), ("Brussels", "BE"), ("Bucharest", "RO"),
    ("Budapest", "HU"), ("Chisinau", "MD"), ("Copenhagen", "DK"),
    ("Dublin", "IE"), ("Helsinki", "FI"), ("Kyiv", "UA"),
    ("London", "GB"), ("Madrid", "ES"), ("Minsk", "BY"),
    ("Moscow", "RU"), ("Oslo", "NO"), ("Paris", "FR"),
    ("Prague", "CZ"), ("Riga", "LV"), ("Rome", "IT"),
    ("Sarajevo", "BA"), ("Sofia", "BG"), ("Stockholm", "SE"),
    ("Vienna", "AT"), ("Warsaw", "PL"), ("Zagreb", "HR"),
    ("Istanbul", "TR"), ("Saint Petersburg", "RU"), ("Hamburg", "DE"),
    ("Munich", "DE"), ("Frankfurt am Main", "DE"), ("Milan", "IT"),
    ("Naples", "IT"), ("Palermo", "IT"), ("Barcelona", "ES"),
    ("Valencia", "ES"), ("Sevilla", "ES"), ("Marseille", "FR"),
    ("Birmingham", "GB"), ("Glasgow", "GB"), ("Kraków", "PL"),
    ("Göteborg", "SE"), ("Odesa", "UA"), ("Kharkiv", "UA"),
]
FEATURES = ["temperature", "rain", "wind_speed", "wind_direction",
            "humidity", "clouds", "visibility", "snow"]

df = pd.read_csv(FILE, dtype={"city_name": "category",
                              "country_code": "category"})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True,
                                 format="ISO8601")
key = df["city_name"].astype(str) + "|" + df["country_code"].astype(str)
position = {f"{n}|{c}": j for j, (n, c) in enumerate(PANEL)}
df["pos"] = key.map(position)
assert df["pos"].notna().all() and df["pos"].nunique() == len(PANEL)
df = df.sort_values(["timestamp", "pos"])

hours = pd.DatetimeIndex(df["timestamp"].unique())
assert (hours[1:] - hours[:-1] == pd.Timedelta(hours=1)).all()
X = df[FEATURES].to_numpy(float).reshape(len(hours), len(PANEL),
                                         len(FEATURES))
coords = df.groupby("pos")[["latitude", "longitude"]].first()
rain = X[:, :, FEATURES.index("rain")]          # (hours, cities)
month = hours.month.to_numpy()
del df, key
print(X.shape, hours[0], "->", hours[-1])

(54096, 45, 8) 2020-01-01 00:00:00+00:00 -> 2026-03-03 23:00:00+00:00


---

## 4. The model: quantiles per city and month

For each city and calendar month, the quantiles of hourly rain over
every hour of the file, at the levels 0.005, 0.015, …, 0.995. About 85% of hours are dry, so most
members are 0 and the top ones spread over the wet-hour amounts: a
zero-inflated distribution, read straight off the data.

In [5]:
M = 100
LEVELS = (np.arange(M) + 0.5) / M

# (12, cities, M): each city's rain quantiles, per calendar month
table = np.stack([np.quantile(rain[month == k], LEVELS, axis=0).T
                  for k in range(1, 13)])
glasgow = table[:, [n for n, _ in PANEL].index("Glasgow")]
print("Glasgow, January: zero members", (glasgow[0] == 0).sum(),
      "| top five", glasgow[0, -5:].round(2))
print("Glasgow, July:    zero members", (glasgow[6] == 0).sum(),
      "| top five", glasgow[6, -5:].round(2))

Glasgow, January: zero members 74 | top five [0.97 1.1  1.4  1.79 2.5 ]
Glasgow, July:    zero members 68 | top five [0.8 1.  1.3 1.8 3. ]


Save it as the file the agent reads, rounded to 0.01 mm:

In [6]:
import json

TABLE_FILE = "rain_quantiles.json"


def save_table(table, path=TABLE_FILE):
    """{"Name|CC": 12 months x M members}, rounded to 0.01 mm."""
    quantiles = {f"{n}|{c}": np.round(table[:, j], 2).tolist()
                 for j, (n, c) in enumerate(PANEL)}
    with open(path, "w") as f:
        json.dump({"members": M, "quantiles": quantiles}, f)
    print(path, f"{os.path.getsize(path) / 1e3:.0f} kB")


save_table(table)

rain_quantiles.json 272 kB


---

## 5. `agent.py`

The challenge imports `agent.py`, builds `Agent()` once with no
argument, and calls `predict`. The table is read from the directory
of `agent.py` itself, where the uploaded files are, never from the
working directory.

It needs nothing but the standard library, so it runs on the lightest
runtime, 181 (Python 3.12 with numpy and pandas). The cities are
looked up by name, not by position, so a change of order in the
request cannot silently mix them up.

In [7]:
%%writefile agent.py
"""European Rain Forecast: city x month climatology, 100 members."""
import json
import os
from datetime import datetime, timedelta

HERE = os.path.dirname(os.path.abspath(__file__))


class Agent:
    def __init__(self):
        with open(os.path.join(HERE, "rain_quantiles.json")) as f:
            self.quantiles = json.load(f)["quantiles"]

    def predict(self, request):
        issue = datetime.fromisoformat(
            request["issue_time"].replace("Z", "+00:00"))
        months = [(issue + timedelta(hours=h)).month
                  for h in request["horizons"]]
        rain = []
        for city in request["cities"]:
            by_month = self.quantiles[
                f"{city['name']}|{city['country']}"]
            rain.append([by_month[m - 1] for m in months])
        return {"rain": rain}

Writing agent.py


---

## 6. Submit

First, one call with the keys `agent.py` reads: a wrong file name or
city key fails here rather than in the challenge's first run.

In [8]:
import importlib

import agent

importlib.reload(agent)             # pick up any edit to agent.py
request = {"issue_time": "2026-03-01T23:00:00Z", "horizons": [6, 48],
           "cities": [{"name": n, "country": c} for n, c in PANEL]}
print(np.asarray(agent.Agent().predict(request)["rain"]).shape)

(45, 2, 100)


`client.submit` creates the submission, uploads the two files, pins
the runtime and deploys it. Without `runtime_id` the challenge's
default runtime is used; 181 is the plain Python one, enough for an
agent that imports nothing but the standard library (182 adds
scikit-learn, LightGBM and XGBoost).

**Run this cell once.** Each run creates a new submission.

In [9]:
result = client.submit(CHALLENGE_ID,
                       files=["agent.py", TABLE_FILE],
                       submission_name="city-month-climatology",
                       runtime_id=181)
submission_id = result["submission_id"]
print(submission_id)

8956


`status` goes `deploy_queue` → `deploy_run` → `active`; a failed
deploy says why in `last_status_message`. Once `active`, the agent
is called at every run, every 6 hours.

In [10]:
status = client.status(submission_id, CHALLENGE_ID)
print(status["status"], "|", status["last_status_message"])
for run in status["run_info"]["results"][-5:]:
    print(run["created_at_ts"], run["job_status"],
          run["submission_reward"])

deploy_queue | Deployment queued successfully
2026-09-21T14:31:36.647250Z pending None
2026-09-21T14:31:36.647250Z pending None


**The first score appears about 48 hours after the first forecast.**
A forecast is scored once both of its hours, +6 h and +48 h, have
happened, so the first eight runs of a new agent report no score
(`None`), not 0. From then on every run scores the forecast made 48
hours earlier. The leaderboard's `MeanReward` is the mean score over
the scored runs, in mm: lower is better.

In [11]:
board = client.leaderboard(CHALLENGE_ID)
if board.empty:
    print("no scored submission yet")
else:
    print(board[["Rank", "SubmissionName", "Username", "MeanReward",
                 "NumberOfRuns"]].head(10).to_string(index=False))

 Rank         SubmissionName       Username  MeanReward  NumberOfRuns
    1       baseline-starter           jojo    0.006564             1
    2       baseline-wet-now           jojo    0.008467             1
    3     baseline-reference           jojo    0.009101             1
    4   baseline-climatology           jojo    0.009367             1
    5 city-month-climatology        raphael         NaN             0
    5 city-month-climatology      funnyvamp         NaN             0
    5 city-month-climatology johnny.li.1411         NaN             0
    5 city-month-climatology        lyne-am         NaN             0
    5 city-month-climatology    anne-sophie         NaN             0
    5 city-month-climatology    hafiz-sanou         NaN             0


---

## 7. Where to go next

The climatology reads no input. The next notebook replays the
challenge on the forecasts issued since 2025-01-01 and scores agents
the way the challenge does: this climatology, a random forest set
against it, and then your own features and model:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-model.ipynb)